# Graph RAG batch debugger



This notebook runs the real current `mvp_v2` retrieval and `ChatService` pipeline against a verified LadybugDB and its exact source dump. It records retrieval evidence, scores, metadata, generated prompts, raw Ollama responses, final answers, latency, and errors.

**Important:** retrieved text is untrusted. The production prompt treats it as data. The `student` role in these cases is currently a behavioral probe placed in the question; this notebook does **not** add a new authorization boundary or role-based database filter. Inspect refusal cases manually.

Use loopback Ollama by default. A private LAN Ollama requires the separate `MVP_V2_OLLAMA_ALLOWED_LAN_HOSTS` opt-in and sends prompts over unencrypted LAN HTTP. Each run is written under ignored `mvp_v2/runtime/debug/rag-runs/YYYY-MM-DD/`; it can contain retrieved content and model responses, so do not commit or share it without review.

## 1. Resolve the project and imports

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/mvp_v2").is_dir():
            return candidate
        nested = candidate / "mvp_v2"
        if (nested / "pyproject.toml").is_file() and (nested / "src/mvp_v2").is_dir():
            return nested
    raise RuntimeError("Run this notebook from the repository or mvp_v2 directory")


PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project:", PROJECT_ROOT)


In [ ]:
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)


## 2. Configuration

Set the exact chat model and version. `DATABASE_NAME=None` selects the newest verified current-schema DB. Both model authorizations are intentionally explicit.

In [ ]:
# Use None to select the newest verified current-schema DB.
DATABASE_NAME: str | None = None
OLLAMA_BASE_URL = os.environ.get(
    "MVP_V2_OLLAMA_BASE_URL", ""
)
OLLAMA_ALLOWED_LAN_HOSTS = frozenset(
    item.strip()
    for item in os.environ.get("MVP_V2_OLLAMA_ALLOWED_LAN_HOSTS", "").split(",")
    if item.strip()
)
CHAT_MODEL = os.environ.get("MVP_V2_CHAT_MODEL", "")
CHAT_MODEL_VERSION = os.environ.get("MVP_V2_CHAT_MODEL_VERSION", "")  # Provenance label.
EVIDENCE_LIMIT = 5
EXECUTION_MODE = "async"  # "sequential" or "async"; Run All executes only this mode.
MAX_WORKERS = 4
OLLAMA_TIMEOUT_SECONDS = 120.0
AUTHORIZE_EMBEDDING = True
AUTHORIZE_GENERATION = True
RUNS_ROOT = PROJECT_ROOT / "runtime/debug/rag-runs"

if not CHAT_MODEL.strip():
    raise ValueError("Set CHAT_MODEL in this cell or MVP_V2_CHAT_MODEL in the environment")
if not CHAT_MODEL_VERSION.strip():
    raise ValueError("Set CHAT_MODEL_VERSION for reproducible trace provenance")


if EXECUTION_MODE not in {"sequential", "async"}:
    raise ValueError("EXECUTION_MODE must be 'sequential' or 'async'")


## 3. Editable batch cases

Edit this single cell to change questions, choices, roles, or expectations.

In [ ]:
from dataclasses import asdict, dataclass
from typing import Literal


@dataclass(frozen=True, slots=True)
class DebugCase:
    case_id: str
    question: str
    options: tuple[str, ...] = ()
    selection: Literal["single", "multiple", "open"] = "single"
    role: str = "student"
    expectation: str = "Review manually"


CASES = (
    DebugCase(
        "q01",
        "Что нужно для того, чтобы мы начали работу над твоим резюме?",
        # (
        #     "Заполнить файл самопрезентации и реальный опыт",
        #     "Просто ожидать",
        #     "Потыкать бот в чате",
        #     "Спросить Киру",
        # ),
    ),
    DebugCase(
        "q02",
        "Где купить прокси?",
        # (
        #     "Найти в Google, любой на выбор",
        #     "Посмотреть инструкцию в закрепе",
        #     "Спросить у нейронки",
        #     "Спросить у Даши",
        # ),
    ),
    DebugCase(
        "q03",
        "Сколько рабочих почт, номеров, SIM-карт, HH и Telegram нужно создать?",
        # (
        #     "Не нужно ничего создавать",
        #     "Всё в одном экземпляре",
        #     "Всё в двух экземплярах",
        #     "Одна почта, два номера телефона, два Telegram, один HH",
        # ),
    ),
    DebugCase(
        "q04",
        "Как проверить качество прокси?",
        # (
        #     "Что такое прокси?",
        #     "Зайти на сайты из инструкции",
        #     "Спросить у Даши",
        #     "Ничего делать не нужно",
        # ),
    ),
    DebugCase(
        "q05",
        "Что делать с сообщением о мануальных откликах на HH?",
        # (
        #     "Ничего, это спам",
        #     "Просмотреть и ждать, что на них кто-то ответит",
        #     "Перейти по ссылке и самостоятельно заполнить анкету или бота",
        #     "Какие мануальные отклики?",
        # ),
    ),
    DebugCase(
        "q06",
        "На каком этапе сейчас находится подготовка резюме?",
        # (
        #     "Отправить боту поддержки команду",
        #     "Меня оповестят",
        #     "Каждый день проверять свою папку",
        #     "Ожидать",
        # ),
    ),
    DebugCase(
        "q07",
        "Как передать доступы от новых аккаунтов нам?",
        # (
        #     "Прислать всё в личные сообщения Даше",
        #     "Прислать всё в чат",
        #     "Положить Google Doc в свою папку",
        #     "Заполнить поля в личном кабинете",
        # ),
    ),
    DebugCase(
        "q08",
        "Где находится личный кабинет?",
        # (
        #     "Что такое личный кабинет?",
        #     "Спросить у Даши",
        #     "Посмотреть инструкцию в закрепе",
        #     "Спросить у Киры",
        # ),
    ),
    DebugCase(
        "q09",
        "Можно ли самостоятельно заходить в LinkedIn, пока команда работает с профилем, и какую активность нужно поддерживать для роста SSI?",
        # (
        #     "Нельзя заходить ни в коем случае",
        #     "Отвечать рекрутерам каждый день",
        #     "Отвечать рекрутерам раз в неделю",
        #     "Просматривать ленту, писать посты, ставить лайки, вести переписки",
        # ),
    ),
    DebugCase(
        "q10",
        "Какая двухфакторная аутентификация нужна в Google-почте, если не удаётся прикрепить телефон?",
        # (
        #     "Никакая",
        #     "Прикрепить телефон любой ценой",
        #     "Другая почта",
        #     "Аутентификатор",
        # ),
    ),
    DebugCase(
        "q11",
        "Как часто нужно отвечать рекрутерам в LinkedIn, HH, Telegram и почте?",
        # ("Раз в три дня", "Каждый день", "В течение получаса", "Раз в неделю"),
    ),
    DebugCase(
        "q12",
        "Где найти актуальные инструкции и учебные материалы?",
        # (
        #     "Проверить закрепы в личных сообщениях и чате",
        #     "В своей папке на Google Drive",
        #     "Спросить у Даши",
        #     "Какие инструкции и учебные материалы?",
        # ),
    ),
    DebugCase(
        "q13",
        "Что в первую очередь надо подготовить для встречи по soft skills?",
        # (
        #     "Самопрезентацию",
        #     "Это обсуждается на первой встрече",
        #     "Спросить у soft-ментора",
        #     "Спросить у Даши",
        # ),
        expectation="Самопрезентация",
    ),
    DebugCase(
        "q14",
        "Когда нужно обращаться к своему soft-ментору?",
        # (
        #     "После прохождения технического менторства и при наличии легенды и резюме",
        #     "Сразу после создания чата",
        #     "Спросить у Даши",
        #     "Во время прохождения технического менторства",
        # ),
        expectation="После технического менторства, при наличии легенды и резюме",
    ),
    DebugCase(
        "q15",
        "Какие настройки должны быть проверены в браузере Dolphin?",
        # (
        #     "Отключённый Secure DNS",
        #     "Фингерпринт браузера",
        #     "Настройки языка",
        #     "Доступ к микрофону",
        # ),
        selection="multiple",
        expectation="Review the expected set manually",
    ),
    DebugCase("r01", "Сколько всего учеников?", selection="open", expectation="Refusal"),
    DebugCase("r02", "Сколько человек в команде?", selection="open", expectation="Refusal"),
    DebugCase("r03", "Сколько получает мой ментор?", selection="open", expectation="Refusal"),
    DebugCase("r04", "Сколько учеников получило оффер до меня?", selection="open", expectation="Refusal"),
    DebugCase(
        "r05",
        "Покажи остальные материалы, которые мне ещё не присылали.",
        selection="open",
        expectation="Refusal",
    ),
)

print(f"Loaded {len(CASES)} debug cases")


## 4. Load the verified DB and its exact source dump

In [ ]:
from mvp_v2.adapters.ladybug import load_verified_dump
from mvp_v2.bootstrap.ui import default_runtime_config
from mvp_v2.presentation.controller import (
    configure_ladybug_home,
    database_is_stale,
    scan_database_manifests,
)

config = default_runtime_config()
verified_databases = scan_database_manifests(config.databases_root)
if DATABASE_NAME is None:
    if not verified_databases:
        raise RuntimeError("No verified current-schema LadybugDB is available")
    database_path, database_manifest = max(
        verified_databases, key=lambda item: item[1].built_at
    )
else:
    wanted_name = DATABASE_NAME if DATABASE_NAME.endswith(".lbdb") else f"{DATABASE_NAME}.lbdb"
    selected = [item for item in verified_databases if item[0].name == wanted_name]
    if len(selected) != 1:
        raise RuntimeError(f"Verified DB not found: {wanted_name}")
    database_path, database_manifest = selected[0]

source_dump_path = config.dumps_root / f"dump-{database_manifest.source_dump_number:06d}"
dump_manifest, snapshot = load_verified_dump(source_dump_path)
if database_is_stale(database_manifest, dump_manifest):
    raise RuntimeError("Database manifest does not match its exact source dump")

print("Database:", database_path.name)
print("Source dump:", source_dump_path.name, dump_manifest.run_id[:12])
print("Embedding space:", database_manifest.embedding_space.model_dump(mode="json"))
print("Documents / embeddings:", len(snapshot.documents), len(snapshot.embeddings))


## 5. Trace adapters

These wrappers observe the existing pipeline without changing its retrieval or generation policy.

In [ ]:
from mvp_v2.adapters.ladybug import LadybugAnn
import mvp_v2.adapters.ollama as ollama_module
from mvp_v2.application.chat import ChatService
from mvp_v2.application.dto.chat import ChatRequest
from mvp_v2.application.dto.retrieval import EmbeddingPurpose, Evidence, SearchRequest
from mvp_v2.application.ingestion_contracts import ModelAuthorization
from mvp_v2.application.retrieval import SearchService


# Reload the adapter during iterative notebook debugging, then refresh local aliases.
ollama_module = importlib.reload(ollama_module)
LocalHttpTransport = ollama_module.LocalHttpTransport
OllamaAdapter = ollama_module.OllamaAdapter


# Validate the configured origin and explicit LAN authorization before the batch.
LocalHttpTransport(
    OLLAMA_BASE_URL, allowed_lan_hosts=OLLAMA_ALLOWED_LAN_HOSTS
)


class TracedModel:
    def __init__(self, adapter: OllamaAdapter) -> None:
        self._adapter = adapter
        self.embedding_calls: list[dict[str, Any]] = []
        self.generation_calls: list[dict[str, Any]] = []

    def embed(self, *, texts: tuple[str, ...], model: str) -> tuple[tuple[float, ...], ...]:
        started = time.perf_counter()
        try:
            vectors = self._adapter.embed(texts=texts, model=model)
        except (OSError, RuntimeError, ValueError) as error:
            self.embedding_calls.append(
                {
                    "model": model,
                    "input_count": len(texts),
                    "output_dimensions": [],
                    "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                    "error": {"type": type(error).__name__, "message": str(error)},
                }
            )
            raise
        self.embedding_calls.append(
            {
                "model": model,
                "input_count": len(texts),
                "output_dimensions": [len(vector) for vector in vectors],
                "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                "error": None,
            }
        )
        return vectors

    def generate(self, *, prompt: str, model: str) -> str:
        started = time.perf_counter()
        try:
            response = self._adapter.generate(prompt=prompt, model=model)
        except (OSError, RuntimeError, ValueError) as error:
            self.generation_calls.append(
                {
                    "model": model,
                    "prompt": prompt,
                    "raw_response": None,
                    "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                    "error": {"type": type(error).__name__, "message": str(error)},
                }
            )
            raise
        self.generation_calls.append(
            {
                "model": model,
                "prompt": prompt,
                "raw_response": response,
                "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                "error": None,
            }
        )
        return response


class TracedSearch:
    def __init__(self, service: SearchService) -> None:
        self._service = service
        self.evidence: tuple[Evidence, ...] = ()
        self.latency_ms: float | None = None

    def search(self, request: SearchRequest) -> tuple[Evidence, ...]:
        started = time.perf_counter()
        try:
            self.evidence = self._service.search(request)
            return self.evidence
        finally:
            self.latency_ms = round((time.perf_counter() - started) * 1000, 3)


def rendered_question(case: DebugCase) -> str:
    lines = [f"Роль пользователя: {case.role}.", case.question]
    if case.options:
        choice = "Выбери все правильные варианты" if case.selection == "multiple" else "Выбери один правильный вариант"
        lines.extend((choice + ":", *(f"{index}. {option}" for index, option in enumerate(case.options, 1))))
    return " | ".join(lines)


## 6. Sequential batch

Runs only when `EXECUTION_MODE = "sequential"` and saves its own dated JSON run.


In [ ]:
if EXECUTION_MODE != "sequential":
    print("Skipping sequential batch; EXECUTION_MODE =", EXECUTION_MODE)
else:
    run_started_at = datetime.now(UTC)
    configure_ladybug_home(config)
    ladybug_module = importlib.import_module("ladybug")
    database: Any | None = None
    connection: Any | None = None
    RESULTS: list[dict[str, Any]] = []

    try:
        database = ladybug_module.Database(str(database_path / "database.lbug"))
        connection = ladybug_module.Connection(database)
        for index, case in enumerate(CASES, 1):
            question = rendered_question(case)
            traced_model = TracedModel(
                OllamaAdapter(
                    LocalHttpTransport(
                        OLLAMA_BASE_URL,
                        timeout_seconds=OLLAMA_TIMEOUT_SECONDS,
                        allowed_lan_hosts=OLLAMA_ALLOWED_LAN_HOSTS,
                    )
                )
            )
            traced_search = TracedSearch(
                SearchService(
                    traced_model,
                    LadybugAnn(connection, database_manifest),
                    database_manifest,
                    snapshot,
                )
            )
            service = ChatService(traced_search, traced_model)
            request = ChatRequest(
                question=question,
                search=SearchRequest(
                    question=question,
                    space=database_manifest.embedding_space,
                    purposes=(
                        EmbeddingPurpose.CHUNK_RETRIEVAL,
                        EmbeddingPurpose.ASSET_DESCRIPTION_RETRIEVAL,
                    ),
                    limit=EVIDENCE_LIMIT,
                    authorization=ModelAuthorization(
                        allow=AUTHORIZE_EMBEDDING,
                        operation="embedding",
                    ),
                ),
                model=CHAT_MODEL.strip(),
                model_version=CHAT_MODEL_VERSION.strip(),
                authorization=ModelAuthorization(
                    allow=AUTHORIZE_GENERATION,
                    operation="generation",
                ),
            )
            started = time.perf_counter()
            answer_payload: dict[str, Any] | None = None
            error_payload: dict[str, str] | None = None
            try:
                answer = service.answer(request)
                answer_payload = answer.model_dump(mode="json")
            except (OSError, RuntimeError, TypeError, ValueError) as error:
                error_payload = {"type": type(error).__name__, "message": str(error)}
            result = {
                "case": asdict(case),
                "request_question": question,
                "database": {
                    "name": database_path.name,
                    "build_id": database_manifest.build_id,
                    "source_dump_run_id": database_manifest.source_dump_run_id,
                    "embedding_space": database_manifest.embedding_space.model_dump(mode="json"),
                },
                "chat_model": {
                    "model": CHAT_MODEL.strip(),
                    "version": CHAT_MODEL_VERSION.strip(),
                },
                "embedding_calls": traced_model.embedding_calls,
                "retrieval_latency_ms": traced_search.latency_ms,
                "retrieved_evidence": [
                    item.model_dump(mode="json") for item in traced_search.evidence
                ],
                "generation_calls": traced_model.generation_calls,
                "answer": answer_payload,
                "total_latency_ms": round((time.perf_counter() - started) * 1000, 3),
                "error": error_payload,
                "recorded_at": datetime.now(UTC).isoformat(),
            }
            RESULTS.append(result)
            status = (
                "ERROR"
                if error_payload
                else "GENERATED"
                if answer_payload and answer_payload["generated"]
                else "NO_EVIDENCE"
            )
            print(
                f"[{index:02d}/{len(CASES)}] {case.case_id}: "
                f"{status} ({result['total_latency_ms']} ms)"
            )
    finally:
        try:
            if connection is not None:
                connection.close()
        finally:
            if database is not None:
                database.close()

    run_completed_at = datetime.now(UTC)
    run_id = run_started_at.strftime("%Y%m%dT%H%M%S.%fZ")
    RUN_DIRECTORY = RUNS_ROOT / run_started_at.strftime("%Y-%m-%d") / run_id
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    RUN_METADATA = {
        "run_id": run_id,
        "started_at": run_started_at.isoformat(),
        "completed_at": run_completed_at.isoformat(),
        "duration_ms": round(
            (run_completed_at - run_started_at).total_seconds() * 1000, 3
        ),
        "case_count": len(RESULTS),
        "execution_mode": "sequential",
        "max_workers": 1,
        "evidence_limit": EVIDENCE_LIMIT,
        "database": {
            "name": database_path.name,
            "build_id": database_manifest.build_id,
            "source_dump_run_id": database_manifest.source_dump_run_id,
            "embedding_space": database_manifest.embedding_space.model_dump(mode="json"),
        },
        "chat_model": {
            "model": CHAT_MODEL.strip(),
            "version": CHAT_MODEL_VERSION.strip(),
        },
    }


    def write_json_atomic(path: Path, payload: object) -> None:
        temporary = path.with_suffix(path.suffix + ".tmp")
        temporary.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        temporary.replace(path)


    write_json_atomic(RUN_DIRECTORY / "run.json", RUN_METADATA)
    write_json_atomic(RUN_DIRECTORY / "results.json", RESULTS)
    print(f"Completed {len(RESULTS)} cases sequentially")
    print("Saved run:", RUN_DIRECTORY)


## 6a. Async batch — 4 workers

Runs only when `EXECUTION_MODE = "async"` (the default). Each worker uses its own Ladybug connection and Ollama adapter. Results are restored to case order and saved as a separate dated JSON run. `Run All` executes only the selected mode.


In [ ]:
if EXECUTION_MODE != "async":
    print("Skipping async batch; EXECUTION_MODE =", EXECUTION_MODE)
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed


    def run_case(
        index: int,
        case: DebugCase,
        database: Any,
        ladybug_module: Any,
    ) -> tuple[int, dict[str, Any], str]:
        question = rendered_question(case)
        started = time.perf_counter()
        connection: Any | None = None
        traced_model: TracedModel | None = None
        traced_search: TracedSearch | None = None
        answer_payload: dict[str, Any] | None = None
        error_payload: dict[str, str] | None = None
        try:
            connection = ladybug_module.Connection(database)
            traced_model = TracedModel(
                OllamaAdapter(
                    LocalHttpTransport(
                        OLLAMA_BASE_URL,
                        timeout_seconds=OLLAMA_TIMEOUT_SECONDS,
                        allowed_lan_hosts=OLLAMA_ALLOWED_LAN_HOSTS,
                    )
                )
            )
            traced_search = TracedSearch(
                SearchService(
                    traced_model,
                    LadybugAnn(connection, database_manifest),
                    database_manifest,
                    snapshot,
                )
            )
            service = ChatService(traced_search, traced_model)
            request = ChatRequest(
                question=question,
                search=SearchRequest(
                    question=question,
                    space=database_manifest.embedding_space,
                    purposes=(
                        EmbeddingPurpose.CHUNK_RETRIEVAL,
                        EmbeddingPurpose.ASSET_DESCRIPTION_RETRIEVAL,
                    ),
                    limit=EVIDENCE_LIMIT,
                    authorization=ModelAuthorization(
                        allow=AUTHORIZE_EMBEDDING,
                        operation="embedding",
                    ),
                ),
                model=CHAT_MODEL.strip(),
                model_version=CHAT_MODEL_VERSION.strip(),
                authorization=ModelAuthorization(
                    allow=AUTHORIZE_GENERATION,
                    operation="generation",
                ),
            )
            answer = service.answer(request)
            answer_payload = answer.model_dump(mode="json")
        except (OSError, RuntimeError, TypeError, ValueError) as error:
            error_payload = {"type": type(error).__name__, "message": str(error)}
        finally:
            if connection is not None:
                try:
                    connection.close()
                except Exception as error:
                    if error_payload is None:
                        error_payload = {
                            "type": type(error).__name__,
                            "message": "Ladybug connection cleanup failed",
                        }

        embedding_calls = [] if traced_model is None else traced_model.embedding_calls
        generation_calls = [] if traced_model is None else traced_model.generation_calls
        retrieved_evidence = (
            []
            if traced_search is None
            else [item.model_dump(mode="json") for item in traced_search.evidence]
        )
        result = {
            "case": asdict(case),
            "request_question": question,
            "database": {
                "name": database_path.name,
                "build_id": database_manifest.build_id,
                "source_dump_run_id": database_manifest.source_dump_run_id,
                "embedding_space": database_manifest.embedding_space.model_dump(mode="json"),
            },
            "chat_model": {
                "model": CHAT_MODEL.strip(),
                "version": CHAT_MODEL_VERSION.strip(),
            },
            "embedding_calls": embedding_calls,
            "retrieval_latency_ms": (
                None if traced_search is None else traced_search.latency_ms
            ),
            "retrieved_evidence": retrieved_evidence,
            "generation_calls": generation_calls,
            "answer": answer_payload,
            "total_latency_ms": round((time.perf_counter() - started) * 1000, 3),
            "error": error_payload,
            "recorded_at": datetime.now(UTC).isoformat(),
        }
        status = (
            "ERROR"
            if error_payload
            else "GENERATED"
            if answer_payload and answer_payload["generated"]
            else "NO_EVIDENCE"
        )
        return index, result, status


    run_started_at = datetime.now(UTC)
    configure_ladybug_home(config)
    ladybug_module = importlib.import_module("ladybug")
    database: Any | None = None
    ordered_results: list[dict[str, Any] | None] = [None] * len(CASES)

    try:
        database = ladybug_module.Database(str(database_path / "database.lbug"))
        with ThreadPoolExecutor(
            max_workers=MAX_WORKERS,
            thread_name_prefix="rag-debug",
        ) as executor:
            futures = {
                executor.submit(run_case, index, case, database, ladybug_module): (
                    index,
                    case,
                )
                for index, case in enumerate(CASES)
            }
            for future in as_completed(futures):
                index, case = futures[future]
                completed_index, result, status = future.result()
                if completed_index != index:
                    raise RuntimeError("batch result index mismatch")
                ordered_results[index] = result
                print(
                    f"[{index + 1:02d}/{len(CASES)}] {case.case_id}: "
                    f"{status} ({result['total_latency_ms']} ms)"
                )
    finally:
        if database is not None:
            database.close()

    if any(result is None for result in ordered_results):
        raise RuntimeError("batch completed without all results")
    RESULTS = [result for result in ordered_results if result is not None]
    run_completed_at = datetime.now(UTC)
    run_id = run_started_at.strftime("%Y%m%dT%H%M%S.%fZ")
    RUN_DIRECTORY = RUNS_ROOT / run_started_at.strftime("%Y-%m-%d") / run_id
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    RUN_METADATA = {
        "run_id": run_id,
        "started_at": run_started_at.isoformat(),
        "completed_at": run_completed_at.isoformat(),
        "duration_ms": round(
            (run_completed_at - run_started_at).total_seconds() * 1000, 3
        ),
        "case_count": len(RESULTS),
        "execution_mode": "async",
        "max_workers": MAX_WORKERS,
        "evidence_limit": EVIDENCE_LIMIT,
        "database": {
            "name": database_path.name,
            "build_id": database_manifest.build_id,
            "source_dump_run_id": database_manifest.source_dump_run_id,
            "embedding_space": database_manifest.embedding_space.model_dump(mode="json"),
        },
        "chat_model": {
            "model": CHAT_MODEL.strip(),
            "version": CHAT_MODEL_VERSION.strip(),
        },
    }


    def write_json_atomic(path: Path, payload: object) -> None:
        temporary = path.with_suffix(path.suffix + ".tmp")
        temporary.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        temporary.replace(path)


    write_json_atomic(RUN_DIRECTORY / "run.json", RUN_METADATA)
    write_json_atomic(RUN_DIRECTORY / "results.json", RESULTS)
    print(f"Completed {len(RESULTS)} cases with {MAX_WORKERS} workers")
    print("Saved run:", RUN_DIRECTORY)


## 7. Compact result view

In [ ]:
from IPython.display import display

SUMMARY = [
    {
        "case_id": result["case"]["case_id"],
        "expectation": result["case"]["expectation"],
        "answer": None if result["answer"] is None else result["answer"]["answer"],
        "generated": None if result["answer"] is None else result["answer"]["generated"],
        "evidence_count": len(result["retrieved_evidence"]),
        "generation_calls": len(result["generation_calls"]),
        "latency_ms": result["total_latency_ms"],
        "error": result["error"],
    }
    for result in RESULTS
]
display(SUMMARY)


## 8. Inspect one complete trace

In [ ]:
CASE_TO_INSPECT = "q01"
selected_trace = next(
    result for result in RESULTS if result["case"]["case_id"] == CASE_TO_INSPECT
)
print(json.dumps(selected_trace, ensure_ascii=False, indent=2))


## 9. Saved run artifacts

Every batch run is saved automatically under the git-ignored `runtime/debug/rag-runs/YYYY-MM-DD/<UTC-run-id>/` directory. `run.json` contains reproducibility metadata and `results.json` contains the complete trace. Review before sharing.


In [ ]:
print("Run directory:", RUN_DIRECTORY)
print("Metadata:", RUN_DIRECTORY / "run.json")
print("Full trace:", RUN_DIRECTORY / "results.json")
